# Smart Fraud Detection Pipeline
### Gold Layer - Fraud Detection and Business Analytics

The Gold layer applies business rules to the trusted Silver datasets to identify potentially fraudulent transactions.

#### Fraud Detection Rule

1. A transaction is classified as FRAUD when its account_id exists in the fraud watchlist.

2. Transactions whose account_id does not appear in the watchlist are classified as NORMAL.

#### Gold Outputs

- Fraud transaction-level dataset
- Overall fraud summary
- Account-level fraud analysis
- Fraud-type analysis

In [0]:
# imports
from pyspark.sql import functions as F

In [0]:
# configuration
CATALOG = "fraud_detection"

SILVER_ENRICHED = f"{CATALOG}.silver.enriched_transactions"
SILVER_FRAUD = f"{CATALOG}.silver.known_fraud_accounts"

GOLD_SCHEMA = f"{CATALOG}.gold"

GOLD_FRAUD_TRANSACTIONS = f"{GOLD_SCHEMA}.fraud_transactions"
GOLD_SUMMARY = f"{GOLD_SCHEMA}.fraud_summary"
GOLD_ACCOUNT_ANALYSIS = f"{GOLD_SCHEMA}.account_fraud_analysis"
GOLD_TYPE_ANALYSIS = f"{GOLD_SCHEMA}.fraud_type_analysis"

In [0]:
# read silver tables
df_transactions = spark.table(SILVER_ENRICHED)
df_fraud_watchlist = spark.table(SILVER_FRAUD)

print("Silver data loaded successfully.")

print("Enriched transactions:", df_transactions.count())
print("Fraud watchlist:", df_fraud_watchlist.count())

Silver data loaded successfully.
Enriched transactions: 200
Fraud watchlist: 10


In [0]:
# inspect fruad watchlist
display(df_fraud_watchlist)

account_id,fraud_type,flagged_date,_source_file,_ingested_at
ACC-00009,IDENTITY_THEFT,2026-01-18,dbfs:/Volumes/fraud_detection/source/raw_files/known_fraud_accounts.csv,2026-08-08T13:12:41.055Z
ACC-00044,MONEY_LAUNDERING,2026-03-28,dbfs:/Volumes/fraud_detection/source/raw_files/known_fraud_accounts.csv,2026-08-08T13:12:41.055Z
ACC-00012,IDENTITY_THEFT,2026-02-07,dbfs:/Volumes/fraud_detection/source/raw_files/known_fraud_accounts.csv,2026-08-08T13:12:41.055Z
ACC-00056,ACCOUNT_TAKEOVER,2026-02-20,dbfs:/Volumes/fraud_detection/source/raw_files/known_fraud_accounts.csv,2026-08-08T13:12:41.055Z
ACC-00055,MONEY_LAUNDERING,2026-01-03,dbfs:/Volumes/fraud_detection/source/raw_files/known_fraud_accounts.csv,2026-08-08T13:12:41.055Z
ACC-00031,PHISHING,2026-03-08,dbfs:/Volumes/fraud_detection/source/raw_files/known_fraud_accounts.csv,2026-08-08T13:12:41.055Z
ACC-00017,ACCOUNT_TAKEOVER,2026-03-08,dbfs:/Volumes/fraud_detection/source/raw_files/known_fraud_accounts.csv,2026-08-08T13:12:41.055Z
ACC-00038,CARD_CLONING,2026-01-13,dbfs:/Volumes/fraud_detection/source/raw_files/known_fraud_accounts.csv,2026-08-08T13:12:41.055Z
ACC-00021,CARD_CLONING,2026-01-14,dbfs:/Volumes/fraud_detection/source/raw_files/known_fraud_accounts.csv,2026-08-08T13:12:41.055Z
ACC-00029,MONEY_LAUNDERING,2026-01-13,dbfs:/Volumes/fraud_detection/source/raw_files/known_fraud_accounts.csv,2026-08-08T13:12:41.055Z


In [0]:
display(
    df_fraud_watchlist.select(
        "account_id",
        "fraud_type",
        "flagged_date"
    )
)

account_id,fraud_type,flagged_date
ACC-00009,IDENTITY_THEFT,2026-01-18
ACC-00044,MONEY_LAUNDERING,2026-03-28
ACC-00012,IDENTITY_THEFT,2026-02-07
ACC-00056,ACCOUNT_TAKEOVER,2026-02-20
ACC-00055,MONEY_LAUNDERING,2026-01-03
ACC-00031,PHISHING,2026-03-08
ACC-00017,ACCOUNT_TAKEOVER,2026-03-08
ACC-00038,CARD_CLONING,2026-01-13
ACC-00021,CARD_CLONING,2026-01-14
ACC-00029,MONEY_LAUNDERING,2026-01-13


**Fraud detection join**

Here, we perfrom transaction and fraud watchlist left join on account_id.

In [0]:
fraud_lookup = (
    df_fraud_watchlist
    .select(
        "account_id",
        "fraud_type",
        "flagged_date"
    )
    .dropDuplicates(["account_id"])
)

In [0]:
df_fraud_detection = (
    df_transactions.alias("t")
    .join(
        fraud_lookup.alias("f"),
        F.col("t.account_id") == F.col("f.account_id"),
        "left"
    )
    .select(
        "t.*",
        F.col("f.fraud_type"),
        F.col("f.flagged_date")
    )
)

**Aplly fraud classification**

The logic is :
if account_id exists in fraud watchlist then FRAUD else NORMAL.

In [0]:
df_fraud_detection = (
    df_fraud_detection
    .withColumn(
        "fraud_status",
        F.when(
            F.col("fraud_type").isNotNull(),
            F.lit("fraud")
        )
        .otherwise(F.lit("normal"))
    )
)

**Add fraud indicator**

In [0]:
df_fraud_detection = (
    df_fraud_detection
    .withColumn(
        "fraud_flag",
        F.when(
            F.col("fraud_status") == "fraud",
            F.lit(1)
        )
        .otherwise(F.lit(0))
    )
)

**calculate transaction risk amount**

In [0]:
df_fraud_detection = (
    df_fraud_detection
    .withColumn(
        "fraud_amount",
        F.when(
            F.col("fraud_flag") == 1,
            F.col("amount")
        )
        .otherwise(F.lit(0.0))
    )
)

**Inspect the result**

In [0]:
display(
    df_fraud_detection.select(
        "txn_id",
        "account_id",
        "txn_date",
        "amount",
        "merchant",
        "fraud_type",
        "flagged_date",
        "fraud_status",
        "fraud_flag",
        "fraud_amount"
    )
)

txn_id,account_id,txn_date,amount,merchant,fraud_type,flagged_date,fraud_status,fraud_flag,fraud_amount
TXN-000001,ACC-00056,2026-01-09,915931.22,UNKNOWN,ACCOUNT_TAKEOVER,2026-02-20,fraud,1,915931.22
TXN-000002,ACC-00025,2026-01-11,23892.36,BookMyShow,null,null,normal,0,0.0
TXN-000003,ACC-00029,2026-02-03,15277.03,Swiggy,MONEY_LAUNDERING,2026-01-13,fraud,1,15277.03
TXN-000004,ACC-00007,2026-03-05,5498.74,ATM,null,null,normal,0,0.0
TXN-000005,ACC-00029,2026-02-23,919.78,Flipkart,MONEY_LAUNDERING,2026-01-13,fraud,1,919.78
TXN-000006,ACC-00047,2026-02-09,22417.98,Swiggy,null,null,normal,0,0.0
TXN-000007,ACC-00003,2026-02-22,3832.61,BigBasket,null,null,normal,0,0.0
TXN-000008,ACC-00037,2026-03-01,11979.58,PhonePe,null,null,normal,0,0.0
TXN-000009,ACC-00035,2026-01-08,23856.18,Unknown_Merchant,null,null,normal,0,0.0
TXN-000010,ACC-00008,2026-03-19,8369.32,BookMyShow,null,null,normal,0,0.0


In [0]:
# validate fraud classification
display(
    df_fraud_detection
    .groupBy("fraud_status")
    .agg(
        F.count("*").alias("transaction_count"),
        F.sum("amount").alias("transaction_amount"),
        F.sum("fraud_amount").alias("fraud_amount")
    )
    .orderBy("fraud_status")
)

fraud_status,transaction_count,transaction_amount,fraud_amount
fraud,26,3807632.109999999,3807632.109999999
normal,174,8516424.86,0.0


In [0]:
# Fraud summary
df_fraud_summary = (
    df_fraud_detection
    .agg(
        F.count("*").alias("total_transactions"),

        F.sum(
            F.when(
                F.col("fraud_status") == "fraud",
                1
            ).otherwise(0)
        ).alias("total_fraud_transactions"),

        F.sum(
            F.when(
                F.col("fraud_status") == "normal",
                1
            ).otherwise(0)
        ).alias("total_normal_transactions"),

        F.sum(
            F.when(
                F.col("fraud_status") == "fraud",
                F.col("amount")
            ).otherwise(0)
        ).alias("total_fraud_amount"),

        F.sum("amount").alias("total_transaction_amount")
    )
    .withColumn(
        "fraud_rate",
        F.round(
            F.col("total_fraud_transactions") /
            F.col("total_transactions") * 100,
            2
        )
    )
)



In [0]:
display(df_fraud_summary)

total_transactions,total_fraud_transactions,total_normal_transactions,total_fraud_amount,total_transaction_amount,fraud_rate
200,26,174,3807632.109999999,1.2324056970000006E7,13.0


### **Account-Level Fraud Analysis**

In [0]:
df_account_fraud = (
    df_fraud_detection
    .groupBy(
        "account_id",
        "customer_name",
        "account_type",
        "branch"
    )
    .agg(
        F.count("*").alias("total_transactions"),

        F.sum("fraud_flag").alias("fraud_transactions"),

        F.sum("amount").alias("total_transaction_amount"),

        F.sum("fraud_amount").alias("total_fraud_amount")
    )
    .withColumn(
        "fraud_rate",
        F.round(
            F.col("fraud_transactions") /
            F.col("total_transactions") * 100,
            2
        )
    )
    .orderBy(
        F.desc("fraud_transactions"),
        F.desc("total_fraud_amount")
    )
)

In [0]:
display(df_account_fraud)

account_id,customer_name,account_type,branch,total_transactions,fraud_transactions,total_transaction_amount,total_fraud_amount,fraud_rate
ACC-00029,Tanuja Nair,SALARY,Pune_FC,6,6,870985.7799999999,870985.7799999999,100.0
ACC-00038,Sushma Thakur,CURRENT,Bangalore_MG,4,4,1863612.85,1863612.85,100.0
ACC-00031,Akash Dhawan,CURRENT,Kolkata_Park,4,4,27582.82,27582.82,100.0
ACC-00044,Vandana Rastogi,CURRENT,Pune_FC,3,3,33524.09,33524.09,100.0
ACC-00017,Ravi Pandey,NRI,Pune_FC,2,2,27505.969999999998,27505.969999999998,100.0
ACC-00021,Sanjay Bhat,NRI,Chennai_T_Nagar,2,2,21764.239999999998,21764.239999999998,100.0
ACC-00009,Venkat Naidu,SAVINGS,Chennai_T_Nagar,2,2,18702.190000000002,18702.190000000002,100.0
ACC-00056,null,null,null,1,1,915931.22,915931.22,100.0
ACC-00012,Rekha Bansal,NRI,Bangalore_MG,1,1,21574.61,21574.61,100.0
ACC-00055,null,null,null,1,1,6448.34,6448.34,100.0


### **Fraud Type Analysis**

In [0]:
df_fraud_type = (
    df_fraud_detection
    .filter(F.col("fraud_status") == "fraud")
    .groupBy("fraud_type")
    .agg(
        F.count("*").alias("fraud_transaction_count"),
        F.sum("amount").alias("fraud_amount"),
        F.avg("amount").alias("average_fraud_amount")
    )
    .orderBy(
        F.desc("fraud_transaction_count")
    )
)

In [0]:
display(df_fraud_type)

fraud_type,fraud_transaction_count,fraud_amount,average_fraud_amount
MONEY_LAUNDERING,10,910958.21,91095.821
CARD_CLONING,6,1885377.0899999999,314229.51499999996
PHISHING,4,27582.82,6895.705
IDENTITY_THEFT,3,40276.8,13425.6
ACCOUNT_TAKEOVER,3,943437.19,314479.0633333333


### **Use spark SQL**

In [0]:
# create temporary view
df_fraud_detection.createOrReplaceTempView("fraud_transactions_view")

In [0]:
fraud_sql_summary = spark.sql("""
SELECT
    fraud_status,
    COUNT(*) AS transaction_count,
    SUM(amount) AS total_amount,
    SUM(fraud_amount) AS fraud_amount
FROM fraud_transactions_view
GROUP BY fraud_status
ORDER BY fraud_status
""")

display(fraud_sql_summary)

fraud_status,transaction_count,total_amount,fraud_amount
fraud,26,3807632.109999999,3807632.109999999
normal,174,8516424.86,0.0


This demonstrates that the same distributed dataset can be analyzed using SQL

In [0]:
high_risk_accounts = spark.sql("""
SELECT
    account_id,
    customer_name,
    account_type,
    branch,
    COUNT(*) AS total_transactions,
    SUM(fraud_flag) AS fraud_transactions,
    SUM(fraud_amount) AS total_fraud_amount,
    ROUND(
        SUM(fraud_flag) * 100.0 / COUNT(*),
        2
    ) AS fraud_rate
FROM fraud_transactions_view
GROUP BY
    account_id,
    customer_name,
    account_type,
    branch
HAVING SUM(fraud_flag) > 0
ORDER BY
    fraud_transactions DESC,
    total_fraud_amount DESC
""")

display(high_risk_accounts)

account_id,customer_name,account_type,branch,total_transactions,fraud_transactions,total_fraud_amount,fraud_rate
ACC-00029,Tanuja Nair,SALARY,Pune_FC,6,6,870985.7799999999,100.00
ACC-00038,Sushma Thakur,CURRENT,Bangalore_MG,4,4,1863612.85,100.00
ACC-00031,Akash Dhawan,CURRENT,Kolkata_Park,4,4,27582.82,100.00
ACC-00044,Vandana Rastogi,CURRENT,Pune_FC,3,3,33524.09,100.00
ACC-00017,Ravi Pandey,NRI,Pune_FC,2,2,27505.969999999998,100.00
ACC-00021,Sanjay Bhat,NRI,Chennai_T_Nagar,2,2,21764.239999999998,100.00
ACC-00009,Venkat Naidu,SAVINGS,Chennai_T_Nagar,2,2,18702.190000000002,100.00
ACC-00056,null,null,null,1,1,915931.22,100.00
ACC-00012,Rekha Bansal,NRI,Bangalore_MG,1,1,21574.61,100.00
ACC-00055,null,null,null,1,1,6448.34,100.00


In [0]:
# create gold schema
spark.sql("""
CREATE SCHEMA IF NOT EXISTS fraud_detection.gold
""")

DataFrame[]

In [0]:
# save fraud transactions
(
    df_fraud_detection
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_FRAUD_TRANSACTIONS)
)

In [0]:
# save fraud summary
(
    df_fraud_summary
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_SUMMARY)
)

In [0]:
# save account analysis
(
    df_account_fraud
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_ACCOUNT_ANALYSIS)
)

In [0]:
# save fraud type analysis
(
    df_fraud_type
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_TYPE_ANALYSIS)
)

In [0]:
# verify tables
spark.sql("""
SHOW TABLES IN fraud_detection.gold
""").show(truncate=False)

+--------+-----------------------+-----------+
|database|tableName              |isTemporary|
+--------+-----------------------+-----------+
|gold    |account_fraud_analysis |false      |
|gold    |fraud_summary          |false      |
|gold    |fraud_transactions     |false      |
|gold    |fraud_type_analysis    |false      |
|        |fraud_transactions_view|true       |
+--------+-----------------------+-----------+

